In [1]:
# =====================================================================
# CELL 1: ENVIRONMENT SETUP
# =====================================================================
!pip install transformers accelerate datasets spacy pillow tqdm torch
!python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\shabnam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.2 MB/s eta 0:00:11
     -- ------------------------------------- 0.8/12.8 MB 1.2 MB/s eta 0:00:11
     --- ------------------------------------ 1.0/12.8 MB 1.2 MB/s eta 0:00:10
     ---- ----------------------------------- 1.3/12.8 MB 1.2 MB/s eta 0:00:10
     ---- ----------------------------------- 1.6/12.8 MB 1.2 MB/s eta 0:00:10
     ----- ---------------------------------- 1.8/12.8 MB 1.2 MB/s eta 0:00:09
     ------ --------------------------------- 2.1/12.8 MB 1.2 MB/s eta 0:00:09
     ------- -------------------------------- 2.4/12.8 MB 1.2 MB/s eta 0:00:09
     -------- ------------------------------- 2.6/12.8 MB 1.2 MB/s eta 0:0


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\shabnam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install -U bitsandbytes transformers accelerate datasets spacy

Defaulting to user installation because normal site-packages is not writeable
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached regex-2026.7.19-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ---------------------------------------- 0.0/39.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/39.1 MB ? eta -:--:--
    --------------------------------------- 0.5/39.1 MB 1.2 MB/s eta 0:00:33
    --------------------------------------- 0.8/39.1 MB 1.2 MB/s eta 0:00:31
   - -------------------------------------- 1.0/39.1 MB 1.2 MB/s eta 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.8.1 requires pyarrow<23,>=4.0.0, but you have pyarrow 25.0.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\shabnam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# =====================================================================
# CELL 2: IMPORTS AND MODEL LOADING (7B MODEL IN 4-BIT)
# =====================================================================
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Loading the massive 7B model
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# 1. Turn on the 4-bit compressor to squeeze it into VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(f"Loading {MODEL_ID} in 4-bit... (This will take a few minutes and use ~5.5GB VRAM)")

# 2. Load model directly onto GPU
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("7B Model successfully loaded in 4-bit! You can now run the pipeline.")


C:\Users\shabnam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# =====================================================================
# CELL 3: PIPELINE FUNCTIONS
# =====================================================================
def extract_visual_claims(text):
    """Extracts short concrete nouns from the reasoning trace using NLP."""
    doc = nlp(text)
    nouns = [chunk.text.lower().strip() for chunk in doc.noun_chunks if len(chunk.text.split()) < 4]
    return list(set(nouns))[:10]

def owl_grounding_score(image, text):
    """Checks if the objects mentioned in the text actually exist in the image."""
    claims = extract_visual_claims(text)
    if not claims:
        return 1.0

    # FIX: max_length=16 prevents both jagged tensors AND exceeds OWL-ViT's token limit
    inputs = owl_processor(
        text=[claims],
        images=image,
        return_tensors="pt",
        padding="max_length",
        max_length=16,
        truncation=True
    ).to("cuda")

    with torch.no_grad():
        outputs = owl_model(**inputs)

    probs = torch.sigmoid(outputs.logits[0])
    max_confidences = probs.max(dim=0).values
    return max_confidences.mean().item()

def extract_final_answer(text):
    """Standardizes answer extraction for majority voting."""
    match = re.search(r'(?:[Tt]he answer is|answer:|choose)\s*([A-D0-9\.\-\/]+)', text)
    if match: return match.group(1).strip()
    nums = re.findall(r'-?\d*\.?\d+', text)
    return nums[-1] if nums else text[-10:].strip()

In [ ]:
# =====================================================================
# CELL 4: CONFIDENCE-WEIGHTED SELF-CONSISTENCY (CISC)
# =====================================================================
def cisc_generate_and_vote(image, question, num_samples=3):
    # 1. VRAM PROTECTION: Resize huge images so attention matrices don't explode
    if max(image.size) > 768:
        ratio = 768 / max(image.size)
        image = image.resize((int(image.size[0] * ratio), int(image.size[1] * ratio)), Image.Resampling.LANCZOS)

    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[prompt], images=[image], return_tensors="pt").to("cuda")

    candidates = []
    answer_votes = defaultdict(float)

    for _ in range(num_samples):
        with torch.no_grad():
            # Temperature must be > 0 for self-consistency to generate diverse paths
            out_ids = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
            reasoning = tokenizer.decode(out_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

            conf_score = owl_grounding_score(image, reasoning)
            ans = extract_final_answer(reasoning)
            answer_votes[ans] += conf_score

            candidates.append({"reasoning": reasoning, "answer": ans, "visual_confidence": conf_score})

        # 2. VRAM PROTECTION: Empty the GPU trash after every single sample
        del out_ids
        torch.cuda.empty_cache()
        gc.collect()

    del inputs
    torch.cuda.empty_cache()

    best_answer = max(answer_votes, key=answer_votes.get)
    best_reasoning = next(c["reasoning"] for c in candidates if c["answer"] == best_answer)
    return best_answer, best_reasoning, dict(answer_votes)

In [ ]:
# =====================================================================
# CELL 5: ACADEMIC BATCH PIPELINE (SYNC-SAFE VERSION)
# =====================================================================
import sys, os, json
from tqdm.auto import tqdm
from datasets import load_dataset

# 1. AUTO-DETECT COLAB VS LOCAL
if 'google.colab' in sys.modules:
    # We are on Colab! Mount Google Drive so progress is saved permanently
    from google.colab import drive
    drive.mount('/content/drive')
    GSV_RESULTS_FILE = "/content/drive/MyDrive/gsv_math_results/cisc_owl_gsv_results.jsonl"
    print("Running on Colab: Saving directly to Google Drive!")
else:
    # We are on your Windows laptop!
    GSV_RESULTS_FILE = "D:/gsv_math_results/cisc_owl_gsv_results.jsonl"
    print("Running locally: Saving to D: Drive!")

os.makedirs(os.path.dirname(GSV_RESULTS_FILE), exist_ok=True)
dataset = load_dataset("AI4Math/MathVista", split="testmini")

completed = set()

# 2. Safely load existing progress line-by-line
if os.path.exists(GSV_RESULTS_FILE):
    with open(GSV_RESULTS_FILE, "r") as f:
        for line in f:
            if line.strip():
                try:
                    data = json.loads(line)
                    completed.add(data["pid"])
                except json.JSONDecodeError:
                    pass

print(f"Resuming: {len(completed)}/1000 academic evaluations complete.")

remaining_samples = [s for s in dataset if s["pid"] not in completed]

# 3. Open file in APPEND mode ("a") to prevent overwriting
with open(GSV_RESULTS_FILE, "a") as f:
    for sample in tqdm(remaining_samples):
        pid = sample["pid"]

        best_ans, best_reasoning, votes = cisc_generate_and_vote(
            sample["decoded_image"],
            sample["query"],
            num_samples=3
        )

        result_dict = {
            "pid": pid,
            "ground_truth": sample["answer"],
            "cisc_final_answer": best_ans,
            "raw_response": best_reasoning,
            "vote_distribution": votes
        }

        # 4. Write exactly one line and FORCE sync to drive!
        f.write(json.dumps(result_dict) + "\n")
        f.flush()
        os.fsync(f.fileno())

print("CISC Grounding Pipeline Complete!")


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…): reconstructing file:   0%|          |  0.00B /  142MB            

data/testmini-00000-of-00001-725687bf7a1(…): downloading bytes:           |  0.00B            

data/test-00000-of-00002-6b81bd7f7e2065e(…): reconstructing file:   0%|          |  0.00B /  358MB            

data/test-00000-of-00002-6b81bd7f7e2065e(…): downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/5141 [00:00<?, ? examples/s]

Resuming: 0/1000 academic evaluations complete.


  0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


KeyboardInterrupt: 